In [ ]:
!git clone -q https://github.com/cad0m/jarida-rag.git

In [ ]:
!pip install -q -e jarida-rag

In [ ]:
# Retrieval only: in-memory search, no LLM and no database, a few minutes per variant
import os
import sys

from kaggle_secrets import UserSecretsClient

DAGSHUB_USER = "cad0m"
secrets = UserSecretsClient()

os.environ["JARIDA_PDF_DIR"] = "/kaggle/input/datasets/ayouberraouy/jarida"
os.environ["JARIDA_WORK"] = "/kaggle/working"
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/jarida-rag.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("DAGSHUB_TOKEN")
# the editable install is only on the path after a restart, so add it for this session
sys.path.insert(0, "/kaggle/working/jarida-rag/src")

import mlflow
import pandas as pd
from jarida_rag import chunk, config, lexical, store
from jarida_rag.evaluation import retrieval

mlflow.set_tracking_uri(config.MLFLOW_URI)
found = mlflow.search_experiments(filter_string="name = 'gazette-rag'")
experiment_id = found[0].experiment_id if found else mlflow.create_experiment("gazette-rag")
mlflow.set_experiment(experiment_id=experiment_id)

embedder = store.load_embedder()
reranker = store.load_reranker()
results, tables = {}, {}


def index(settings):
    for key, value in settings.items():
        setattr(config, key, value)
    chunks = chunk.chunk_all()
    return chunks, store.embed(embedder, [c["embed_text"] for c in chunks])


def run(name, chunks, vectors, lexical_index=None):
    def candidates(question):
        return store.candidates_in_memory(chunks, vectors, embedder, question, lexical_index)

    retriever = "hybrid" if lexical_index else "dense"
    for label, rerank in [(retriever, None), (retriever + "+rerank", reranker)]:
        key = f"{name} {label}"
        tables[key], results[key] = retrieval.evaluate(candidates, rerank, f"retrieval-{name}-{label}", label)

In [ ]:
# 1. Chunking, decided in lab-v3 (short header + unwrap) at 2 chunks per text; this re-checks it at the current cap. Skip to save time
CHUNKING = {
    "old":             {"CHUNK_HEADER": "first_200", "UNWRAP_LINES": False, "SPLIT_PREAMBLE": False},
    "old+unwrap":      {"CHUNK_HEADER": "first_200", "UNWRAP_LINES": True,  "SPLIT_PREAMBLE": False},
    "short-header":    {"CHUNK_HEADER": "short",     "UNWRAP_LINES": False, "SPLIT_PREAMBLE": False},
    "+unwrap-lines":   {"CHUNK_HEADER": "short",     "UNWRAP_LINES": True,  "SPLIT_PREAMBLE": False},
    "+split-preamble": {"CHUNK_HEADER": "short",     "UNWRAP_LINES": True,  "SPLIT_PREAMBLE": True},
}
for name, settings in CHUNKING.items():
    run(name, *index(settings))

In [ ]:
# 2. Retrieval on the chosen chunking: chunks per text before the reranker, then BM25 fused with dense search.
# Keep a change only if, with the reranker, evidence hits rise by 3+ questions and doc hits don't drop.
chunks, vectors = index({"CHUNK_HEADER": "short", "UNWRAP_LINES": True, "SPLIT_PREAMBLE": False})
lexical_index = lexical.build(chunks)
chosen_cap = config.MAX_CHUNKS_PER_TEXT
for cap in (2, 4):
    config.MAX_CHUNKS_PER_TEXT = cap
    run(f"cap{cap}", chunks, vectors)
    run(f"cap{cap}", chunks, vectors, lexical_index)
config.MAX_CHUNKS_PER_TEXT = chosen_cap

In [ ]:
# counts, not 3 decimals: one question is 0.013, so gaps under ~3 questions are noise
total = len(next(iter(tables.values())))
pd.DataFrame({
    name: {
        "doc_hit": f"{round(m['doc_hit_rate'] * total)}/{total}",
        "evidence_hit": f"{round(m['evidence_hit_rate'] * total)}/{total}",
        "evidence_mrr": round(m["evidence_mrr"], 3),
        "seconds": round(m["avg_search_seconds"], 2),
    }
    for name, m in results.items()
}).T

In [ ]:
# what the best row still misses, and where: "not in candidates" is coverage, the other stages are ranking
best = max(results, key=lambda name: results[name]["evidence_mrr"])
print(best, tables[best]["miss_stage"].value_counts().to_dict())
tables[best].query("evidence_rank == 0")[["question", "miss_stage", "candidate_rank", "capped_rank", "retrieved"]]